# E3.3 · Sequencing the programme

**Function E — AI Governance for Agentic Systems → Running the Programme — the CISO Office**  ·  *Security of AI*

Builds on **[E3.2 · Governing autonomy rather than approving tools](https://spbreed.github.io/cyber-commons/lessons/E3.2.html)**.

| | |
|---|---|
| Open-source tooling | — |
| Open-weight models | — |
| Frontier models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

Everything in this programme depends on something else in it, and starting in the wrong order means the first two quarters produce nothing anyone can see. Sequencing is the difference between a programme and a backlog.

> **At CyberTravels.** Everything in TripBot's programme depends on something else in it. Start with the register and the identity work, or the first two quarters produce nothing anyone can see.

## 2 · The framework

```
   dependency order

   inventory ---> tiering ---> control mapping ---> verification
       |                                              ^
       +--> identity + telemetry ---------------------+

   start at the right-hand side and two quarters produce nothing visible
```

Sequencing decides whether the programme compounds or thrashes, and there is a
right order that is not the exciting one:

1. **Inventory** — you cannot govern what you cannot list.
2. **Identity** — agents distinct from humans, separately revocable.
3. **Containment** — egress, paths, tools; deny by default.
4. **Evidence** — log the acting identity; retain enough to replay.
5. **Evaluation** — accuracy against a held-out key, per release.
6. **Continuous** — drift alerts and freshness windows on every control.

The popular order inverts it, because evaluation and dashboards demo well and
identity does not. Doing 5 before 2 produces a well-measured system nobody can
switch off — and that is not a hypothetical, it is the modal state of AI
security programmes.

Each step also unlocks the next: you cannot log the acting identity (4) before
agents have identities (2), and you cannot alert on drift (6) without a baseline
from evidence (4).

## 3 · Demo — the dependency graph, and what each step unlocks

In [ ]:
STEPS = {
 1: ("inventory",   [],     ["you can now tier and assign owners"]),
 2: ("identity",    [1],    ["per-agent revocation", "attribution in logs"]),
 3: ("containment", [1],    ["bounded blast radius", "a red team has something to test"]),
 4: ("evidence",    [2],    ["act chains", "replayable runs", "a drift baseline"]),
 5: ("evaluation",  [4],    ["accuracy you can defend", "regression cases"]),
 6: ("continuous",  [4,5],  ["freshness windows", "drift alerts", "live posture"]),
}
print(f"{'step':>5}  {'name':14s}{'needs':10s}unlocks")
print("-" * 84)
for n, (name, needs, unlocks) in STEPS.items():
    print(f"{n:>5}  {name:14s}{str(needs):10s}{'; '.join(unlocks)}")

def can_do(step, done):
    return all(d in done for d in STEPS[step][1])

print("\nwhat is doable from a standing start:")
print("   ", [n for n in STEPS if can_do(n, set())])

## 4 · Where it breaks — evaluation first

In [ ]:
def simulate(order):
    done, blocked, timeline = set(), [], []
    for step in order:
        if can_do(step, done):
            done.add(step); timeline.append((step, STEPS[step][0], "done"))
        else:
            missing = [STEPS[d][0] for d in STEPS[step][1] if d not in done]
            blocked.append((step, STEPS[step][0], missing))
            timeline.append((step, STEPS[step][0], f"BLOCKED on {missing}"))
    return done, blocked, timeline

POPULAR = [5, 6, 1, 3, 2, 4]     # evaluation and dashboards first
CORRECT = [1, 2, 3, 4, 5, 6]

for label, order in (("popular order", POPULAR), ("correct order", CORRECT)):
    done, blocked, timeline = simulate(order)
    print(f"=== {label} ===")
    for step, name, state in timeline:
        print(f"   {step}. {name:14s}{state}")
    print(f"   completed {len(done)}/6, blocked {len(blocked)}\n")

done_pop, blocked_pop, _ = simulate(POPULAR)
print(f"popular order completes {len(done_pop)}/6 on the first pass;")
print(f"{len(blocked_pop)} step(s) have to be redone after their prerequisites land.")
assert len(blocked_pop) > 0

## 5 · The control — measure the programme by capability, not activity

In [ ]:
CAPABILITY = {
 1: "can list every AI asset with an owner",
 2: "can revoke one agent without stopping the others",
 3: "can bound what a compromised agent reaches",
 4: "can say who caused a specific action, and replay it",
 5: "can defend an accuracy number to a supervisor",
 6: "can say what is TRUE TODAY, not what passed once",
}
def programme_state(done):
    return [(n, CAPABILITY[n], n in done) for n in STEPS]

for label, order in (("eval-first, one quarter in", POPULAR[:2]),
                     ("correct order, one quarter in", CORRECT[:3])):
    done, _, _ = simulate(order)
    print(f"=== {label} — {len(done)} capabilit(y/ies) ===")
    for n, cap, have in programme_state(done):
        print(f"   {'YES' if have else 'no ':4s} {cap}")
    print()

done_a, _, _ = simulate(POPULAR[:2])
done_b, _, _ = simulate(CORRECT[:3])
print(f"after equal effort: eval-first has {len(done_a)} capabilities, "
      f"correct order has {len(done_b)}")
print("\nThe eval-first programme can produce a dashboard. It cannot switch")
print("anything off, and it cannot say who did what.")
assert len(done_b) > len(done_a)

## What you just proved

Only inventory is doable from a standing start. The popular evaluation-first order completes 4 of 6 on the first pass with 2 steps blocked on missing prerequisites; the correct order completes all six. After equal effort the eval-first programme holds 1 capability against the correct order's 3, and can produce a dashboard while being unable to revoke an agent or attribute an action.

## Your turn

Locate your programme on the six steps honestly. Most are between 2 and 3 while reporting on 5, which is exactly the gap this sequence prevents — and the fix is to stop reporting 5 until 2 and 3 are done.

---

**Next → [E3.4 · Org design and ownership](https://spbreed.github.io/cyber-commons/lessons/E3.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E3.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E3.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*